In [3]:
import pandas as pd
import time
import ast
df = pd.read_csv('../data/Hotel_Reviews.csv')
def replace_address(row):
    if "Netherlands" in row["Hotel_Address"]:
        return "Amsterdam, Netherlands"
    elif "Barcelona" in row["Hotel_Address"]:
        return "Barcelona, Spain"
    elif "United Kingdom" in row["Hotel_Address"]:
        return "London, United Kingdom"
    elif "Milan" in row["Hotel_Address"]:        
        return "Milan, Italy"
    elif "France" in row["Hotel_Address"]:
        return "Paris, France"
    elif "Vienna" in row["Hotel_Address"]:
        return "Vienna, Austria" 

# Replace all the addresses with a shortened, more useful form
df["Hotel_Address"] = df.apply(replace_address, axis = 1)
# The sum of the value_counts() should add up to the total number of reviews
print(df["Hotel_Address"].value_counts())

Hotel_Address
London, United Kingdom    262301
Barcelona, Spain           60149
Paris, France              59928
Amsterdam, Netherlands     57214
Vienna, Austria            38939
Milan, Italy               37207
Name: count, dtype: int64


In [4]:
display(df.groupby("Hotel_Address").agg({"Hotel_Name":"nunique"}))

,Hotel_Name
Hotel_Address,
"Amsterdam, Netherlands",105
"Barcelona, Spain",211
"London, United Kingdom",400
"Milan, Italy",162
"Paris, France",458
"Vienna, Austria",158


In [11]:
import time
import pandas as pd
start = time.time()
df = pd.read_csv('../data/Hotel_Reviews.csv')

In [8]:
df.drop(["lat", "lng"], axis = 1, inplace = True)

In [9]:
df["Hotel_Address"] = df.apply(replace_address, axis = 1)

In [10]:
df.drop(["Additional_Number_of_Scoring"], axis = 1, inplace = True)
df.Total_Number_of_Reviews = df.groupby("Hotel_Name")["Total_Number_of_Reviews"].transform("count")
df.Average_Score = round(df.groupby("Hotel_Name")["Reviewer_Score"].transform("mean"), 1)


In [20]:
# Process the Tags into new columns
# The file Hotel_Reviews_Tags.py, identifies the most important tags
# Leisure trip, Couple, Solo traveler, Business trip, Group combined with Travelers with friends, 
# Family with young children, Family with older children, With a pet
df["Leisure_trip"] = df.Tags.apply(lambda tag: 1 if "Leisure trip" in tag else 0)
df["Couple"] = df.Tags.apply(lambda tag: 1 if "Couple" in tag else 0)
df["Solo_traveler"] = df.Tags.apply(lambda tag: 1 if "Solo traveler" in tag else 0)
df["Business_trip"] = df.Tags.apply(lambda tag: 1 if "Business trip" in tag else 0)
df["Group"] = df.Tags.apply(lambda tag: 1 if "Group" in tag or "Travelers with friends" in tag else 0)
df["Family_with_young_children"] = df.Tags.apply(lambda tag: 1 if "Family with young children" in tag else 0)
df["Family_with_older_children"] = df.Tags.apply(lambda tag: 1 if "Family with older children" in tag else 0)
df["With_a_pet"] = df.Tags.apply(lambda tag: 1 if "With a pet" in tag else 0)

In [16]:
import time 
import pandas as pd
import nltk as nltk
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
df = pd.read_csv('../data/Hotel_Reviews_Filtered.csv')
sia = SentimentIntensityAnalyzer()
stop_words = stopwords.words('english')
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()
    tokens =nltk.word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    return " ".join(tokens)
    # Apply sentiment analysis using VADER (on raw text here)
df['Positive_Review_Sentiment'] = df['Positive_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])
df['Negative_Review_Sentiment'] = df['Negative_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Optional: combine positive and negative sentiments for an overall score
df['Overall_Sentiment'] = df[['Positive_Review_Sentiment', 'Negative_Review_Sentiment']].mean(axis=1)

# Save the dataframe with new NLP data
print("Saving results to Hotel_Reviews_NLP.csv")
df.to_csv('../data/Hotel_Reviews_NLP.csv', index=False)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Saving results to Hotel_Reviews_NLP.csv


In [19]:
from nltk.corpus import stopwords
df = pd.read_csv("../data/Hotel_Reviews_Filtered.csv")
start = time.time()
cache = set(stopwords.words('english'))
def remove_stopwords(review):
    text = "".join([word for word in review.split() if word not in cache])
    return text
df.Negative_Review = df.Negative_Review.apply(remove_stopwords)
df.Positive_Review = df.Positive_Review.apply(remove_stopwords)

In [20]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vader_sentiment = SentimentIntensityAnalyzer()
def calc_sentiment(review):
    if review == "No Negative" or review == "No Positive":
        return 0
    return vader_sentiment.polarity_scores(review)["compound"]

In [21]:
print("Calculating sentiment columns for both positive and negative reviews")
start = time.time()
df["Negative_Sentiment"] = df.Negative_Review.apply(calc_sentiment)
df["Positive_Sentiment"] = df.Positive_Review.apply(calc_sentiment)
end = time.time()
print("Calculationg sentiment took" + str(round(end-start, 2)) + " Seconds")

Calculating sentiment columns for both positive and negative reviews
Calculationg sentiment took27.21 Seconds


In [23]:
df = df.sort_values(by = ["Negative_Sentiment"], ascending=True)
print(df[["Negative_Review", "Negative_Sentiment"]])
df = df.sort_values(by=["Positive_Sentiment"], ascending = True)
print(df[["Positive_Review", "Positive_Sentiment"]])

       Negative_Review  Negative_Sentiment
349307        Cheating             -0.5574
52956         Horrible             -0.5423
114472             bad             -0.5423
478167             Bad             -0.5423
147464             bad             -0.5423
...                ...                 ...
317800           Great              0.6249
397663         Awesome              0.6249
482922           great              0.6249
478966           Great              0.6249
119192           great              0.6249

[515738 rows x 2 columns]
       Positive_Review  Positive_Sentiment
235836        disaster             -0.6249
501482             Bad             -0.5423
409738             bad             -0.5423
427819             bad             -0.5423
209542             bad             -0.5423
...                ...                 ...
429765            Love              0.6369
36292             BEST              0.6369
326446            best              0.6369
362320            Love     

In [24]:
df = df.reindex(["Hotel_Name", "Hotel_Address", "Total_Number_of_Reviews", "Average_Score", "Reviewer_Score", "Negative_Sentiment", "Positive_Sentiment", "Reviewer_Nationality", "Leisure_trip", "Couple", "Solo_traveler", "Business_trip", "Group", "Family_with_young_children", "Family_with_older_children", "With_a_pet", "Negative_Review", "Positive_Review"], axis = 1)
print("Saving results to Hotel_Reviews_NLP.csv")
df.to_csv(r"../data/Hotel_Reviews_NLP.csv", index= False)

Saving results to Hotel_Reviews_NLP.csv
